# An Encoder–Decoder Network for Neural Machine Translation

In [4]:
## Download the spanish/english dataset

url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
path = tf.keras.utils.get_file("spa-eng.zip", origin=url, extract=True) # cache_dir defaults to ~/.keras/datasets


from pathlib import Path
# The file might be extracted to a subdirectory within ~/.keras/datasets
# Search for 'spa.txt' in subdirectories:
dataset_path = Path(path).parent
for file_path in dataset_path.rglob('spa.txt'):
    if file_path.is_file():
        dataset_path = file_path
        break

text = dataset_path.read_text()

In [5]:
import numpy as np

text = text.replace("¡", "").replace("¿", "")
pairs = [line.split("\t") for line in text.splitlines()]
np.random.shuffle(ßpairs)
sentences_english, sentences_spanish = zip(*pairs)  # separates the pairs into 2 lists

In [6]:
for i in range(3):
    print(sentences_english[i], "=>", sentences_spanish[i])

Tom has a reputation of never letting anyone else say anything. => Tom tiene reputación de nunca dejar a nadie más decir nada.
I want to leave Paris. => Quiero dejar París.
Please shuffle the cards carefully. => Por favor, baraja las cartas cuidadosamente.


In [7]:
vocab_size = 12000
max_length = 50

# create a keras TextVectorization layer using the above parameters
# then call .adapt on that layer, passing in the english sentences as parameter

layer_english_vectorization = tf.keras.layers.TextVectorization(
    vocab_size, output_sequence_length=max_length, pad_to_max_tokens=False, ragged=False, name = "English_Vectorize")
layer_english_vectorization.adapt(sentences_english)

# now do the same with the spanish -- make and adapt a layer
# BUT for the spanish add 'starttoken' and 'endtoken' to each sentence first

layer_spanish_vectorization = tf.keras.layers.TextVectorization(
    vocab_size, output_sequence_length=max_length,  pad_to_max_tokens=False, ragged=False, name = "Spanish_Vectorize")
layer_spanish_vectorization.adapt([f"starttoken {s} endtoken" for s in sentences_spanish])

I0000 00:00:1776103167.914699   47251 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8766 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


In [8]:
layer_english_vectorization.get_vocabulary()[:10]

['', '[UNK]', 'the', 'i', 'to', 'you', 'tom', 'a', 'is', 'he']

In [9]:
layer_spanish_vectorization.get_vocabulary()[:10]

['', '[UNK]', 'starttoken', 'endtoken', 'de', 'que', 'a', 'no', 'tom', 'la']

In [10]:
layer_spanish_vectorization('endtoken')[0].numpy()

3

In [11]:
len(sentences_english)

118964

In [12]:
X_train_encoder = tf.constant(sentences_english[:100_000])
X_valid_encoder = tf.constant(sentences_english[100_000:])
X_train_decoder = tf.constant([f"starttoken {s}" for s in sentences_spanish[:100_000]])
X_valid_decoder = tf.constant([f"starttoken {s}" for s in sentences_spanish[100_000:]])
Y_train = layer_spanish_vectorization([f"{s} endtoken" for s in sentences_spanish[:100_000]])
Y_valid = layer_spanish_vectorization([f"{s} endtoken" for s in sentences_spanish[100_000:]])